In [49]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.tools import tool
from typing import Optional
from pathlib import Path
from dotenv import load_dotenv

import sqlite3

In [50]:
def _resolve_db_path() -> str:
    # Search upward from cwd so the notebook works whether started at repo root or notebooks/.
    for current in [Path.cwd(), *Path.cwd().parents]:
        candidate = current / "data" / "processed" / "electric_data_table.db"
        if candidate.exists():
            return str(candidate.resolve())
    raise FileNotFoundError("Could not find data/processed/electric_data_table.db from current working directory.")

DB_PATH = _resolve_db_path()
print(f"Using DB_PATH: {DB_PATH}")

Using DB_PATH: D:\Development\intelligent-actuator-data-agent\data\processed\electric_data_table.db


In [52]:
#TOOL 1:
@tool
def query_by_part_number(part_number: str) -> str:
    """
    Consults the database to retrieve all technical specifications of a specific actuator 
    using its exact base part number. Use this tool when the user mentions a specific model code.
    """

    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    clean_part_number = part_number.strip().replace("*", "")

    query = 'SELECT * FROM electric_data_table WHERE "Base Part Number" LIKE ?'
    cursor.execute(query, (f"%{clean_part_number}%",))
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return f"No actuator found for part number: {part_number}"

    return str([dict(row) for row in rows])

In [53]:
#TOOL 2:
@tool
def query_by_spects(
    enclosure_type: Optional[str] = None,
    voltage: Optional[str] = None,
    phase: Optional[str] = None,
    min_on_off_torque: Optional[float] = None,
    min_modulating_torque: Optional[int] = None,
    max_speed_seconds: Optional[int] = None,
    motor_power: Optional[int] = None
) -> str:
    """
    Search and filter the actuators database based on strict engineering requirements.

    Args:
        enclosure_type: 'Weatherproof' or 'Explosionproof'.
        voltage: Operating voltage, e.g., '110V', '220V', '24V'.
        phase: Electrical phase number, typical values are 1 or 3.
        min_on_off_torque: Minimum required torque for On/Off operation (Nm).
        min_modulating_torque: Minimum required torque for Modulating operation (Nm).
        max_speed_seconds: Maximum acceptable operating time to open/close (seconds).
        motor_power: Minimum required motor power in watts.
    """

    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    query = "SELECT * FROM electric_data_table WHERE 1=1"
    params = []

    if enclosure_type:
        query += ' AND "Enclosure Type" = ?'
        params.append(enclosure_type)

    if voltage:
        query += ' AND "Voltage" = ?'
        params.append(voltage)

    if phase is not None:
        query += ' AND "Phase" = ?'
        params.append(phase)

    if min_on_off_torque is not None:
        query += ' AND "On/Off Output Torque Nm" >= ?'
        params.append(min_on_off_torque)

    if min_modulating_torque is not None:
        query += ' AND "Modulating Output Torque Nm" >= ?'
        params.append(min_modulating_torque)

    if max_speed_seconds is not None:
        # Match actuators that are equal or faster (lower seconds) than requested.
        query += ' AND "Operating Speed (sec) 60 Hz" <= ?'
        params.append(max_speed_seconds)

    if motor_power is not None:
        query += ' AND "Motor Power Watts" >= ?'
        params.append(motor_power)

    query += " LIMIT 5"

    cursor.execute(query, params)
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return "No actuators found matching the specified filters."

    return str([dict(row) for row in rows])

In [54]:
tools = [query_by_part_number, query_by_spects]

In [2]:
agent = create_agent(
    model=ChatOpenAI(
        model="gpt-5-mini",
        temperature=0.1,
    ),
    system_prompt = """
You are the Konecto Intelligent Actuator Data Agent. Your job is to assist clients 
in finding and specifying the correct actuator models from the Series 76 catalog.
You have tools to query the local SQLite database. Always use them when asked about specific technical data.
"""
)



In [62]:
agent= create_agent(
    model=ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.1),
    tools=tools,
    system_prompt = """
You are the Konecto Intelligent Actuator Data Agent. Your job is to assist clients 
in finding and specifying the correct actuator models from the Series 76 catalog.
You have tools to query the local SQLite database. Always use them when asked about specific technical data.
"""
)

result = agent.invoke({"messages": [{"role": "user", "content": "I need an actuator that the enclosure type is weatherproof"}]})
print(result["messages"][-1].content)

[{'type': 'text', 'text': 'Here are the weatherproof actuators from the Series 76 catalog:\n\n*   **Base Part Number:** 761A00-11300000/A\n    *   **Enclosure Type:** Weatherproof\n    *   **Voltage:** 110V\n    *   **Phase:** Single\n    *   **On/Off Output Torque Nm:** 80.0\n    *   **Modulating Output Torque Nm:** 65.0\n    *   **Operating Speed (sec) 60 Hz:** 17.0\n    *   **Motor Power Watts:** 15.0\n\n*   **Base Part Number:** 761B00-11300000/A\n    *   **Enclosure Type:** Weatherproof\n    *   **Voltage:** 110V\n    *   **Phase:** Single\n    *   **On/Off Output Torque Nm:** 100.0\n    *   **Modulating Output Torque Nm:** 80.0\n    *   **Operating Speed (sec) 60 Hz:** 17.0\n    *   **Motor Power Watts:** 15.0\n\n*   **Base Part Number:** 762A00-11300000/A\n    *   **Enclosure Type:** Weatherproof\n    *   **Voltage:** 110V\n    *   **Phase:** Single\n    *   **On/Off Output Torque Nm:** 150.0\n    *   **Modulating Output Torque Nm:** 130.0\n    *   **Operating Speed (sec) 60 Hz:

In [56]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("PRAGMA table_info(electric_data_table)")
cols = cursor.fetchall()
conn.close()

for c in cols:
    print(c[1])

Enclosure Type
Voltage
Phase
Base Part Number
On/Off Output Torque In-Lbs.
On/Off Output Torque Nm
On/Off Duty Cycle S4%
On/Off Cycles per Hour
Modulating Output Torque In-Lbs.
Modulating Output Torque Nm
Modulating Duty Cycle S4%
Modulating Starts per Hour
Operating Speed (sec) 60 Hz
Operating Speed (sec) 50 Hz
Full Load Current (FLA) [A] 60 Hz
Full Load Current (FLA) [A] 50 Hz
Locked Rotor Current (LRA) [A] 60 Hz
Locked Rotor Current (LRA) [A] 50 Hz
Motor Power Watts
